# Worked block analysis — CAN transceiver (Hamilton DAG)

**Now driven by Hamilton.** The same CAN transceiver analysis as before, but the leaf inputs and derived nodes live in proper Python modules (`blocks/can_transceiver/leaves.py` and `analysis.py`) and Hamilton wires them into a DAG by parameter name. The notebook is reduced to *loading the project*, *running the DAG*, and *rendering / spec-checking the results*.

**The block.** A CAN bus transceiver on a 5 V rail. Quantities of interest:

1. **Supply power** — mode-dependent current draw × rail tolerance.
2. **Junction temperature** — `ambient + dissipation × R_θJA`. Ambient is auto-supplied from project scenarios.
3. **Spec checks** — power ≤ 500 mW and T_J ≤ 125 °C across every (scenario × mode) combination, with bounds linked to Jama requirement IDs.

**Pipeline:**

```
scenarios.toml ─┐
modes.toml     ─┤
requirements.py┤───▶ Project.load() ──▶ project.run([leaves, analysis], targets=[...])
blocks/...     ─┘                                          │
                                                           ▼
                                                results: dict[str, Quantity]
                                                           │
                                                           ▼
                                                  rendering / spec checks
```

## 1. Load project + block modules

`Project.load(...)` reads `project/scenarios.toml` and `project/modes.toml` and auto-extracts every context key that appears in *every* scenario (here: `ambient_temp` and `vbat`) into Quantity inputs ready for the DAG.

In [1]:
import sys
from pathlib import Path

# Make `blocks/`, `project/` (here) and `components/` (one dir up) importable.
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))

from framework import Project, get_contract_meta, requirements
from framework.units import V, A, mA, mW, K, degC

project = Project.load(
    scenarios=Path.cwd() / "project" / "scenarios.toml",
    modes=Path.cwd() / "project" / "modes.toml",
)

# Importing project.requirements auto-populates framework.requirements.
from project.requirements import OPERATING_TEMP, CAN_5V_RAIL, VBAT

# Block subpackage: leaves + analysis (DAG nodes), contracts (public commitments),
# verifications (formal spec checks). Other blocks should
# `from blocks.can_transceiver import can_5v_draw` rather than reach into the
# private modules; the framework's cycle detector enforces this at project.run
# time.
from blocks.can_transceiver import leaves, analysis, contracts, verifications
from components.instances.semiconductors.tja1051t_3 import TJA1051T_3

print(f"Project: {len(project.scenarios)} scenarios, "
      f"{len(project.modes)} modes, "
      f"{len(requirements.list_all())} requirements")
print(f"Scenarios: {project.scenarios.names()}")
print(f"Modes:     {project.modes.names()}")
print(f"Auto-extracted DAG inputs from project state: "
      f"{[k for k in project.standard_inputs() if k not in ('scenarios', 'modes')]}")
print(f"Active part: {TJA1051T_3.part_number} ({TJA1051T_3.description})")

Project: 3 scenarios, 4 modes, 9 requirements
Scenarios: ['nominal', 'cold_low_vin', 'hot_high_vin']
Modes:     ['off', 'sleep', 'active', 'diagnostic']
Auto-extracted DAG inputs from project state: ['ambient_temp', 'vbat']
Active part: TJA1051T/3 (NXP high-speed CAN transceiver, 5V supply with 3.3V I/O level)


## 2. The block layout

Per design doc 6.6 a block subpackage looks like:

```
blocks/can_transceiver/
  __init__.py     # re-exports public Contracts ONLY
  leaves.py       # i_supply, v_supply, r_theta_ja, t_j_max  — private to block
  analysis.py    # power, thermal_rise, t_j                   — private to block
  contracts.py    # can_5v_draw                                — PUBLIC, importable
```

Hamilton builds the DAG from the function signatures: a parameter named `v_supply` wires to the function whose name is `v_supply`. The framework's cycle detector (design doc 7.4) enforces a separate rule on top: **cross-block dependencies must flow through Contracts only**. Anything in `leaves` / `analysis` is private to this block; only `contracts.py` is meant to be reachable from elsewhere.

In [2]:
import inspect

modules = [leaves, analysis, contracts]
print("DAG nodes:", project.list_nodes(modules))
print()

# Inspect each block module to see the function shapes Hamilton sees.
for module, name in ((leaves, "leaves"), (analysis, "analysis"), (contracts, "contracts")):
    print(f"=== blocks/can_transceiver/{name}.py ===")
    fns = [m for m in dir(module) if callable(getattr(module, m)) and not m.startswith("_")]
    for fn_name in fns:
        fn = getattr(module, fn_name)
        if getattr(fn, "__module__", None) != module.__name__:
            continue  # skip re-exports
        sig = inspect.signature(fn)
        params = ", ".join(sig.parameters)
        meta = get_contract_meta(fn)
        marker = "  [CONTRACT]" if meta else ""
        print(f"  def {fn_name}({params}) -> Quantity{marker}")
    print()

DAG nodes: ['ambient_temp', 'can_5v_draw', 'can_5v_rail', 'i_supply', 'power', 'r_theta_ja', 't_j', 't_j_max', 'thermal_rise', 'v_supply']

=== blocks/can_transceiver/leaves.py ===
  def i_supply() -> Quantity
  def r_theta_ja() -> Quantity
  def t_j_max() -> Quantity
  def v_supply(can_5v_rail) -> Quantity

=== blocks/can_transceiver/analysis.py ===
  def power(v_supply, i_supply) -> Quantity
  def t_j(ambient_temp, thermal_rise) -> Quantity
  def thermal_rise(power, r_theta_ja) -> Quantity

=== blocks/can_transceiver/contracts.py ===
  def can_5v_draw() -> Quantity  [CONTRACT]



## 3. Execute the DAG

`project.run(modules, targets, inputs)` builds the Hamilton driver from the block modules, layers the user-supplied inputs over the project's standard inputs (`ambient_temp`, `vbat`, `scenarios`, `modes`), and computes the requested targets.

`can_5v_rail` is passed explicitly because the `v_supply` leaf needs the `CAN_5V_RAIL` requirement (REQ-PWR-005) — that's how the block ties its supply tolerance to the project's rail spec.

In [3]:
results = project.run(
    modules=[leaves, analysis, contracts],
    targets=["power", "thermal_rise", "t_j", "t_j_max", "can_5v_draw", "i_supply"],
    inputs={"can_5v_rail": CAN_5V_RAIL},
)

print("Computed targets:")
for name, q in results.items():
    print(f"  {name:14s} unit = {q.unit}")

Computed targets:
  power          unit = mW
  thermal_rise   unit = K
  t_j            unit = °C
  t_j_max        unit = °C
  can_5v_draw    unit = mA
  i_supply       unit = A


## 4. Power dissipation result

`power` carries the mode axis (from `i_supply`) with a range per mode (from `v_supply × i_supply` interval arithmetic).

In [4]:
power = results["power"]
mode_names = project.modes.names()
scen_names = project.scenarios.names()

print(f"{'mode':12s} {'min':>10s} {'max':>10s}")
print("-" * 36)
for m in mode_names:
    val = power.at(mode=m)
    lo, hi = val if isinstance(val, tuple) else (val, val)
    print(f"{m:12s} {lo:9.3f}mW {hi:9.3f}mW")

mode                min        max
------------------------------------
off              0.000mW     0.000mW
sleep            0.038mW     0.079mW
active         213.750mW   341.250mW
diagnostic     332.500mW   472.500mW


## 5. Junction temperature result

`t_j` varies along **both axes simultaneously** — `ambient_temp` brought in the scenario axis, `thermal_rise` brought in the mode axis. Every (scenario × mode) cell has its own min/max range.

In [5]:
t_j = results["t_j"]

def cell_str(val):
    if isinstance(val, tuple):
        lo, hi = val
        return f"{lo:5.1f} – {hi:5.1f} °C"
    return f"{val:5.1f} °C"

print(f"{'mode':12s} | " + " | ".join(f"{s:^16s}" for s in scen_names))
print("-" * (14 + 19 * len(scen_names)))
for m in mode_names:
    cells = [cell_str(t_j.at(mode=m, scenario=s)) for s in scen_names]
    print(f"{m:12s} | " + " | ".join(f"{c:^16s}" for c in cells))

mode         |     nominal      |   cold_low_vin   |   hot_high_vin  
-----------------------------------------------------------------------
off          |      25.0 °C     |     -40.0 °C     |      85.0 °C    
sleep        |  25.0 –  25.0 °C | -40.0 – -40.0 °C |  85.0 –  85.0 °C
active       |  50.6 –  65.9 °C | -14.4 –   0.9 °C | 110.6 – 125.9 °C
diagnostic   |  64.9 –  81.7 °C |  -0.1 –  16.7 °C | 124.9 – 141.7 °C


## 6. Spec checks via `@verification_test`

Step 9a wraps the manual `.within(...)` checks in a proper test type. Each test is a function in `blocks/can_transceiver/verifications.py` carrying its name, requirement ID, severity, and assertion logic. The framework's runner discovers them by module scan, hands each a `VerificationContext` that pulls Quantities out of the DAG results, and returns a `TestResult` per test with the exact (scenario, mode) corners where it failed.

The same definition becomes the source for: pytest in CI (step 9b), Jama push, the design-review report, and the PR-comment summary (step 9c).

In [6]:
from framework import run_verifications, results_to_html_table, results_to_pr_comment
from IPython.display import HTML, Markdown, display

test_results = run_verifications([verifications], results)

# Rich notebook table (step 9c). Same data also feeds the PR-comment markdown
# below and the Jama-shape JSON export — one definition, many destinations.
display(HTML(results_to_html_table(test_results)))

# Drill into the thermal failure: framework hands back the exact (scenario, mode)
# corners — no manual iteration required.
thermal = test_results["t_j stays below block-derated max"]
if not thermal.passed:
    print("T_J failures:")
    for corner in thermal.failed_at:
        t_j_val = thermal.evidence["t_j"].at(mode=corner.mode, scenario=corner.scenario)
        t_j_hi = t_j_val[1] if isinstance(t_j_val, tuple) else t_j_val
        t_j_max_val = thermal.evidence["t_j_max"].at()
        print(f"  {corner}  T_J max = {t_j_hi:5.1f} °C  (over by {t_j_hi - t_j_max_val:.1f} °C)")

# PR-comment formatter — what the bot would post on a pull request.
print("\n--- PR comment preview ---")
display(Markdown(results_to_pr_comment(test_results, title="CAN transceiver verification")))

Verdict,Severity,Test,Failed at
FAIL,critical,t_j stays below block-derated max,"scenario='hot_high_vin', mode='active'; scenario='hot_high_vin', mode='diagnostic'"
PASS,critical,i_supply stays within published can_5v_draw contract,—


T_J failures:
  scenario='hot_high_vin', mode='active'  T_J max = 125.9 °C  (over by 0.9 °C)
  scenario='hot_high_vin', mode='diagnostic'  T_J max = 141.7 °C  (over by 16.7 °C)

--- PR comment preview ---


## CAN transceiver verification
**1 pass**, **1 fail**, **0 warn**, **0 info**

<details><summary>FAIL — t_j stays below block-derated max</summary>

- Severity: `critical`
- Failed at: scenario='hot_high_vin', mode='active'; scenario='hot_high_vin', mode='diagnostic'
- Evidence:
  - `t_j`
  - `t_j_max`

</details>

In [7]:
can_5v_draw = results["can_5v_draw"]
meta = get_contract_meta(contracts.can_5v_draw)
assert meta is not None

print(f"Contract: {contracts.can_5v_draw.__name__}")
print(f"  description: {meta.description}")
print(f"  requirement: {meta.requirement}")
print(f"  assumed_inputs: {meta.assumed_inputs}")
print(f"  compares_to:    {meta.compares_to}  (step 8 run-time consistency target)")
print()
print(f"Declared draw by mode (in {can_5v_draw.unit}):")
for m in mode_names:
    val = can_5v_draw.at(mode=m)
    lo, hi = val if isinstance(val, tuple) else (val, val)
    print(f"  {m:11s} {lo:7.3f} – {hi:7.3f}")

Contract: can_5v_draw
  description: Block's draw from the 5V CAN rail
  requirement: REQ-PWR-005
  assumed_inputs: {'can_5v_rail_window_V': (4.75, 5.25)}
  compares_to:    i_supply  (step 8 run-time consistency target)

Declared draw by mode (in mA):
  off           0.000 –   0.000
  sleep         0.000 –   0.020
  active        0.000 –  75.000
  diagnostic    0.000 – 100.000


## 7. Published contracts

`contracts.py` is this block's public face. Every function there carries a `ContractMeta` (description + Jama requirement + assumed inputs) and the framework's cycle detector — automatically called by `project.run` — guarantees that a downstream block consuming a Contract from here can't accidentally reach past it into `leaves.py` or `analysis.py`.

Right now there's only this one block, so the Contract has no consumer yet. Once the eventual `blocks/power_supply` lands, it will `from blocks.can_transceiver import can_5v_draw` and roll it into its total-load math.

## 8. Where this is going

This notebook used to hand-compute power and junction temp inline. Now it loads a project, imports a block subpackage, asks Hamilton to wire the functions into a DAG, executes against a content-addressed cache, validates the cross-block dependency rules, runs formal verification tests with rich per-corner reporting — and the leaf inputs themselves come from a Pydantic-validated component instance.

What's landed:

- **Block layout** (step 4a). `leaves.py` / `analysis.py` / `contracts.py` / `verifications.py` as proper Python modules.
- **Content-addressed caching** (step 4b). Re-running this notebook hits the cache for every node whose inputs haven't changed.
- **Typed components** (step 5). The CAN transceiver's mode-dependent supply current and thermal resistance come from `TJA1051T_3` — a Pydantic-validated component instance whose fields can themselves carry per-scenario variation.
- **Contracts + cycle detection** (step 7). The `can_5v_draw` Contract is this block's public face; the framework refuses to run any project whose Contracts reach into the private internals of other blocks.
- **Run-time contract consistency** (step 8). `compares_to="i_supply"` ties the declared bound to the block's actual computed draw; `project.run` verifies actual ⊆ declared on every run and raises `ContractViolation` on mismatch.
- **VerificationTest foundation** (step 9a). `@verification_test` + `TestResult` + `run_verifications` — what used to be a manual `.within(...)` block is now a discoverable, named, requirement-linked test type.
- **pytest plugin** (step 9b). `tests/test_verifications.py` parametrizes over every block's `@verification_test`s; CI fails per-test rather than per-suite. Severity → `pytest.fail` / `xfail` / `skip` mapping.
- **Output channels** (step 9c). `results_to_html_table` (above), `results_to_pr_comment` (above), `results_to_markdown_table`, `results_to_jama_records` — one verification-test definition, four destinations.

Still to come:

- Step 6: netlist parser (Protel ASCII) — bind refdes to component instances declared in `components/`. Deferred until a real netlist sample is on hand.
- Cross-block `assumed_inputs` validation — auto-generate a `@verification_test` per Contract's `assumed_inputs` entry once a second block lands.